# Tool · Track many videos at once

**What this notebook does:** The same as Notebook 1, but for **every video in a folder**, one after the other.
Use it when you have more than a handful of videos. It writes the same `_keypoints.csv` files as Notebook 1, so you
can continue with Notebook 2 for each video.

*Run it like the other notebooks: click a code cell and press **Shift + Enter**. You only need to change the Settings.*

## Step 1 · Load the tools

In [ ]:
from concurrent.futures import ThreadPoolExecutor   # lets us track several videos at the same time
from pathlib import Path
import urllib.request

import cv2
import mediapipe as mp
from mediapipe.tasks.python import BaseOptions, vision
import pandas as pd
from tqdm.auto import tqdm

## Step 2 · Settings

In [ ]:
VIDEOS_FOLDER = "../Videos"        # where your videos are
OUTPUT_FOLDER = "../Keypoints"     # where the keypoints files go

# Files with these endings count as videos (upper or lower case doesn't matter)
VIDEO_ENDINGS = [".mov", ".mp4", ".avi", ".mkv", ".m4v"]

MODEL = "full"                     # "lite", "full" or "heavy", as in Notebook 1
MIN_CONFIDENCE = 0.4               # as in Notebook 1
SAVE_OVERLAY_VIDEO = False         # also save videos with the skeleton drawn on? (slower, big files)

SKIP_DONE = True                   # skip videos that already have a keypoints file

# How many videos to track at the same time. 1 is safest. On a computer with many processor cores
# (or a computing cluster), 2-8 can finish a big batch much sooner, but needs more memory.
VIDEOS_AT_ONCE = 1

## Step 3 · Find the videos and the model

In [ ]:
output_folder = Path(OUTPUT_FOLDER)
output_folder.mkdir(exist_ok=True)

videos = sorted(p for p in Path(VIDEOS_FOLDER).iterdir() if p.suffix.lower() in VIDEO_ENDINGS)
if SKIP_DONE:
    videos = [v for v in videos if not (output_folder / f"{v.stem}_keypoints.csv").exists()]
print(f"{len(videos)} videos to track:", ", ".join(v.name for v in videos))

model_path = Path(f"../Notebook/pose_landmarker_{MODEL}.task")
if not model_path.exists():
    print(f"Downloading the '{MODEL}' model. This only happens once...")
    url = ("https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
           f"pose_landmarker_{MODEL}/float16/latest/pose_landmarker_{MODEL}.task")
    urllib.request.urlretrieve(url, model_path)

## Step 4 · The recipe for one video

This *function* does everything Notebook 1 does for one video. See Notebook 1 for a step-by-step explanation.

In [ ]:
POINT_NAMES = [point.name.lower() for point in vision.PoseLandmark]
COLUMNS = ["time_ms"] + [name + axis for name in POINT_NAMES for axis in ["_x", "_y"]]
POINT_STYLE = vision.drawing_utils.DrawingSpec(color=(0, 0, 255), thickness=-1, circle_radius=5)
LINE_STYLE = vision.drawing_utils.DrawingSpec(color=(255, 255, 255), thickness=3)
SKELETON = vision.PoseLandmarksConnections.POSE_LANDMARKS


def track_video(video_path):
    options = vision.PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(model_path)),
        running_mode=vision.RunningMode.VIDEO,
        min_pose_detection_confidence=MIN_CONFIDENCE,
        min_pose_presence_confidence=MIN_CONFIDENCE,
        min_tracking_confidence=MIN_CONFIDENCE,
    )
    video = cv2.VideoCapture(str(video_path))
    fps = video.get(cv2.CAP_PROP_FPS)
    if fps == 0:
        return f"{video_path.name}: could not open this video, skipped"
    if SAVE_OVERLAY_VIDEO:
        size = (int(video.get(cv2.CAP_PROP_FRAME_WIDTH)), int(video.get(cv2.CAP_PROP_FRAME_HEIGHT)))
        overlay = cv2.VideoWriter(str(output_folder / f"{video_path.stem}_overlay.mp4"),
                                  cv2.VideoWriter_fourcc(*"mp4v"), fps, size)

    rows = []
    frame_number = 0
    with vision.PoseLandmarker.create_from_options(options) as tracker:
        while True:
            ok, frame = video.read()
            if not ok:
                break
            time_ms = frame_number * 1000 / fps
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = tracker.detect_for_video(image, round(time_ms))

            row = {"time_ms": time_ms}
            if result.pose_landmarks:
                body = result.pose_landmarks[0]
                for name, point in zip(POINT_NAMES, body):
                    row[name + "_x"] = point.x
                    row[name + "_y"] = point.y
                if SAVE_OVERLAY_VIDEO:
                    vision.drawing_utils.draw_landmarks(frame, body, SKELETON, POINT_STYLE, LINE_STYLE)
            rows.append(row)
            if SAVE_OVERLAY_VIDEO:
                overlay.write(frame)
            frame_number += 1

    video.release()
    if SAVE_OVERLAY_VIDEO:
        overlay.release()

    keypoints = pd.DataFrame(rows, columns=COLUMNS)
    keypoints.to_csv(output_folder / f"{video_path.stem}_keypoints.csv", index=False)
    found = keypoints["nose_x"].notna().mean() * 100
    return f"{video_path.name}: {len(keypoints)} frames, person found in {found:.1f}%"

## Step 5 · Track all videos

The progress bar counts finished videos. A message is printed for each video when it's done.

In [ ]:
with ThreadPoolExecutor(max_workers=VIDEOS_AT_ONCE) as pool:
    for message in tqdm(pool.map(track_video, videos), total=len(videos), unit="video"):
        print(message)